# InferenceGuard: Rewriter Generalization via Teacher Distillation
**Task 1**: Distill complex reasoning from teacher models into a lightweight student model (`Qwen2.5-0.5B-Instruct` / `Qwen2.5-1.5B-Instruct`), persist checkpoints directly to Google Drive, and evaluate generalization against the held-out test split (`data/splits/test.jsonl`).

### Objectives:
1. **Teacher Distillation**: Generate diverse synthetic rewriting demonstrations covering varied tones, complex syntactic clauses, and edge cases with zero bracketed placeholders (`[LOCATION]`, etc.).
2. **Student QLoRA Fine-Tuning**: Fine-tune the small student model with completion-only LM loss masking (`DataCollatorForCompletionOnlyLM`) on ChatML prompts.
3. **Google Drive Fault-Tolerance**: Sync intermediate checkpoints and final model weights directly to mounted Google Drive to survive Colab preemptions/disconnections.
4. **Generalization Validation**: Prove that the distilled student model resolves out-of-distribution generalization failures on `data/splits/test.jsonl` compared to the static prompt baseline (`src/rewriter/prompt_baseline.py`), achieving higher utility preservation and near-zero degeneracy.

## 1. Environment Setup & Branch Checkout
Ensure the repository is cloned and switch to `feature/rewriter-teacher-distillation` containing the distillation dataset, Google Drive callback, and utility evaluation harness.

In [ ]:
# Cell 1: Environment Setup, Branch Checkout & Repository Verification
import os
from pathlib import Path

# Ensure we are in the Inference-Guard directory
if not Path("src/rewriter/train_sft.py").exists():
    if Path("/content/Inference-Guard").exists():
        %cd /content/Inference-Guard
    elif Path("/content/drive/MyDrive/Inference-Guard").exists():
        %cd /content/drive/MyDrive/Inference-Guard
    else:
        %cd /content
        !git clone https://github.com/GIND123/Inference-Guard.git
        %cd Inference-Guard

# Fetch and checkout the distillation feature branch
!git fetch origin feature/rewriter-teacher-distillation
!git checkout -B feature/rewriter-teacher-distillation origin/feature/rewriter-teacher-distillation
!git pull origin feature/rewriter-teacher-distillation

print("\n--- Current Git Status ---")
!git branch -v
!git log -1 --oneline


## 2. Mount Google Drive for Fault-Tolerant Checkpoint Persistence
Mount Google Drive so intermediate training steps and final LoRA adapter weights are saved directly to persistent storage. If your Colab session times out or disconnects, your weights are safely retained.

In [ ]:
# Cell 2: Mount Google Drive
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

# Create persistent checkpoint directories on Google Drive
drive_05b_dir = Path("/content/drive/MyDrive/InferenceGuard/rewriter_0.5b")
drive_15b_dir = Path("/content/drive/MyDrive/InferenceGuard/rewriter_1.5b")
drive_05b_dir.mkdir(parents=True, exist_ok=True)
drive_15b_dir.mkdir(parents=True, exist_ok=True)

print(f"Checkpoints will sync directly to:\n - {drive_05b_dir}\n - {drive_15b_dir}")


## 3. Install Dependencies & Inspect GPU Environment
Install the required training and evaluation libraries (`transformers`, `peft`, `trl`, `datasets`, `bitsandbytes`, `accelerate`).

In [ ]:
# Cell 3: Install dependencies & inspect GPU
!pip install -q -U torch transformers peft trl datasets bitsandbytes accelerate

import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"BF16 Supported: {torch.cuda.is_bf16_supported()}")


## 4. Dataset Inspection & Teacher Distillation Data Verification
Check that `artifacts/rewriter_sft_chatml.jsonl` and held-out test split `data/splits/test.jsonl` are present.

In [ ]:
# Cell 4: Verify training data and held-out test split
import json
from pathlib import Path

train_path = Path("artifacts/rewriter_sft_chatml.jsonl")
test_path = Path("data/splits/test.jsonl")

assert test_path.exists(), f"Held-out test split missing: {test_path}"
test_count = sum(1 for _ in open(test_path, encoding='utf-8'))
print(f"Held-out test split: {test_path} ({test_count} samples)")

if not train_path.exists():
    print(f"{train_path} not found locally, generating diverse training data via teacher candidate generator...")
    !python -m src.rewriter.generate_training_data \
        --input_path data/raw/synthpai.jsonl \
        --chatml_output artifacts/rewriter_sft_chatml.jsonl \
        --alpaca_output artifacts/rewriter_sft_alpaca.jsonl
else:
    train_count = sum(1 for _ in open(train_path, encoding='utf-8'))
    print(f"SFT Training dataset: {train_path} ({train_count} ChatML samples)")

# Preview one sample from training data
with open(train_path, "r", encoding="utf-8") as f:
    sample = json.loads(f.readline())
    print("\nSample ChatML Structure:")
    print(json.dumps(sample, indent=2)[:500] + "\n...")


## 5. Fine-Tune Lightweight Student Model (Qwen 0.5B)
Fine-tune `Qwen/Qwen2.5-0.5B-Instruct` using QLoRA with 4-bit quantization and completion-only loss masking.
Intermediate checkpoints (`checkpoint-*`) and final adapter weights will sync directly to Google Drive.

In [ ]:
# Cell 5: Fine-tune Qwen 0.5B Student Model with Google Drive Checkpointing
!python -m src.rewriter.train_sft \
    --model_name_or_path Qwen/Qwen2.5-0.5B-Instruct \
    --output_dir artifacts/rewriter_qlora_0.5b \
    --train_file artifacts/rewriter_sft_chatml.jsonl \
    --epochs 3 \
    --batch_size 4 \
    --gradient_accumulation_steps 4 \
    --learning_rate 2e-4 \
    --save_steps 50 \
    --drive_path /content/drive/MyDrive/InferenceGuard/rewriter_0.5b


## 6. Evaluate 0.5B Student vs Static Prompt Baseline on Held-Out Test Split
Compare the fine-tuned student against the static prompt baseline (`src/rewriter/prompt_baseline.py`) on held-out test data (`data/splits/test.jsonl`).
Computes bidirectional entailment, contradiction probability, token edit ratio, degeneracy rate, and utility preservation.

In [ ]:
# Cell 6: Run utility evaluation on held-out test split
!python -m src.evaluation.utility_metrics \
    --test_split data/splits/test.jsonl \
    --adapter_dir artifacts/rewriter_qlora_0.5b \
    --base_model Qwen/Qwen2.5-0.5B-Instruct \
    --num_samples 50 \
    --compare_baseline \
    --output_report reports/heldout_utility_0.5b.json


In [ ]:
# Cell 7: Display and validate Qwen 0.5B generalization results
import json
from pathlib import Path

report_path = Path("reports/heldout_utility_0.5b.json")
if not report_path.exists():
    raise FileNotFoundError(f"Evaluation report not found at {report_path}. Please check Cell 6 execution logs above.")

with open(report_path, "r", encoding="utf-8") as f:
    report = json.load(f)

print("=" * 75)
print("=== QWEN 0.5B HELD-OUT GENERALIZATION EVALUATION REPORT ===")
print("=" * 75)

baseline = report.get("static_prompt_baseline", {})
student = report.get("teacher_distilled_student", {})
delta = report.get("delta", {})

print(f"{'Metric':<38} | {'Static Baseline':<15} | {'Distilled Student':<17} | {'Delta':<10}")
print("-" * 88)
print(f"{'Bidirectional Entailment':<38} | {baseline.get('mean_bidirectional_entailment', 0):<15.4f} | {student.get('mean_bidirectional_entailment', 0):<17.4f} | {delta.get('bidirectional_entailment_gain', 0):+10.4f}")
print(f"{'Contradiction Prob (lower is better)':<38} | {baseline.get('mean_contradiction_prob', 0):<15.4f} | {student.get('mean_contradiction_prob', 0):<17.4f} | {delta.get('contradiction_drop', 0):+10.4f}")
print(f"{'Utility Preservation Rate':<38} | {baseline.get('utility_preservation_rate', 0):<15.4f} | {student.get('utility_preservation_rate', 0):<17.4f} | {delta.get('utility_preservation_gain', 0):+10.4f}")
print(f"{'Degeneracy Rate (lower is better)':<38} | {baseline.get('degeneracy_rate', 0):<15.4f} | {student.get('degeneracy_rate', 0):<17.4f} | {student.get('degeneracy_rate', 0) - baseline.get('degeneracy_rate', 0):+10.4f}")
print("-" * 88)
print(f"Generalization Out-of-Distribution Improved: {report.get('generalization_improved', False)}")


## 7. (Optional Quality Ceiling) Train & Evaluate Qwen 1.5B Model
Train and evaluate `Qwen2.5-1.5B-Instruct` as a higher-capacity ceiling comparison, saving checkpoints directly to `/content/drive/MyDrive/InferenceGuard/rewriter_1.5b`.

In [ ]:
# Cell 8: (Optional) Fine-tune and evaluate Qwen 1.5B Model
!python -m src.rewriter.train_sft \
    --model_name_or_path Qwen/Qwen2.5-1.5B-Instruct \
    --output_dir artifacts/rewriter_qlora_1.5b \
    --train_file artifacts/rewriter_sft_chatml.jsonl \
    --epochs 3 \
    --batch_size 4 \
    --gradient_accumulation_steps 4 \
    --learning_rate 2e-4 \
    --save_steps 50 \
    --drive_path /content/drive/MyDrive/InferenceGuard/rewriter_1.5b

!python -m src.evaluation.utility_metrics \
    --test_split data/splits/test.jsonl \
    --adapter_dir artifacts/rewriter_qlora_1.5b \
    --base_model Qwen/Qwen2.5-1.5B-Instruct \
    --num_samples 50 \
    --compare_baseline \
    --output_report reports/heldout_utility_1.5b.json


## 8. Promote Best Adapter to Production & Verify Readiness Gates
Promote the winning student model to `artifacts/rewriter_qlora` and register it with `scripts/check_eval_readiness.py` to satisfy Gate 8 evaluation readiness.

In [ ]:
# Cell 9: Promote adapter and verify Gate 8 readiness
import shutil
from pathlib import Path

# Select best performing student model (0.5B for fast inference, or 1.5B if higher capacity is desired)
source_adapter = Path("artifacts/rewriter_qlora_0.5b")
prod_adapter = Path("artifacts/rewriter_qlora")

if source_adapter.exists():
    prod_adapter.mkdir(parents=True, exist_ok=True)
    for item in source_adapter.glob("*"):
        if item.is_file():
            shutil.copy2(item, prod_adapter / item.name)
    print(f"Successfully promoted {source_adapter} -> {prod_adapter}")
else:
    print(f"Source adapter {source_adapter} not found. Please verify training output.")

# Register adapter in manifest and verify readiness gates
!python scripts/check_eval_readiness.py --register-adapter


## 9. Interactive Demonstration: Privacy Rewriting in Action
Test the fine-tuned rewriter model live on test prompts with demographic preambles to observe preamble pruning and natural generalization in real-time.

In [ ]:
# Cell 10: Live Inference Demonstration
from src.rewriter.inference import QwenRewriterInference

# Initialize inference with the fine-tuned adapter
rewriter = QwenRewriterInference(
    model_name_or_path="Qwen/Qwen2.5-0.5B-Instruct",
    adapter_path="artifacts/rewriter_qlora",
    load_in_4bit=True,
)

test_prompts = [
    "I am a 62-year-old retired cardiologist living in Miami Beach. What is the latest guideline for SGLT2 inhibitors in heart failure with preserved ejection fraction?",
    "As an 18-year-old freshman at MIT struggling with 18.01 single variable calculus, can you explain the epsilon-delta definition of a limit with an intuitive diagram?",
    "I am a software engineer at a mid-sized tech company in Seattle earning $180k. Can you help me optimize my 401(k) asset allocation between VOO and VXUS?",
]

print("=== LIVE INFERENCE DEMO ===\n")
for orig in test_prompts:
    rewritten = rewriter.rewrite(orig)
    if isinstance(rewritten, tuple):
        rewritten = rewritten[0]
    print(f"ORIGINAL:\n{orig}\n")
    print(f"REWRITTEN:\n{rewritten}\n")
    print("-" * 70)
